# IWI experiment · ViT-S/16 (SSL4EO-L, MoCo) on Landsat, NDVI and wavelet bands: IWI and 384 features per cluster

This notebook repeats 09 with the vision transformer of torchgeo's `ViTSmall16_Weights.LANDSAT_ETM_SR_MOCO` in place of ResNet-18: ViT-S/16 (timm `vit_small_patch16_224`) pretrained with MoCo on Landsat 7 ETM+ surface reflectance (SSL4EO-L), the model of `30_ReTraining_Torchgeo` and `31_DeepFeature_Landsat_Loop`. It predicts the wealth (IWI) of a DHS cluster from the nine Landsat-based inputs of 09, and saves for every cluster the 384 values of the layer before the output: the class token after the final layer norm, which is what the output layer reads. One row per `cluster_id`.

| key | input | bands |
|---|---|---|
| `landsat` | Landsat images | 6 SR + TEMP1 |
| `landsat_ndvi` | Landsat images and temporal MODIS NDVI | 6 SR + TEMP1 + 12 monthly NDVI |
| `landsat_wavelet6` | Landsat images and 6 statistical wavelet-based features | 6 SR + TEMP1 + LFC and HFC minimum, maximum and mean |
| `landsat_hfc_max` | Landsat images and max values of HFC | 6 SR + TEMP1 + HFC max |
| `landsat_hfc_mean` | Landsat images and mean values of HFC | 6 SR + TEMP1 + HFC mean |
| `landsat_hfc_min` | Landsat images and min values of HFC | 6 SR + TEMP1 + HFC min |
| `landsat_lfc_max` | Landsat images and max values of LFC | 6 SR + TEMP1 + LFC max |
| `landsat_lfc_mean` | Landsat images and mean values of LFC | 6 SR + TEMP1 + LFC mean |
| `landsat_lfc_min` | Landsat images and min values of LFC | 6 SR + TEMP1 + LFC min |

* **Pretrained bands:** the six surface-reflectance bands BLUE, GREEN, RED, NIR, SWIR1 and SWIR2 are the bands of the weights (ETM+ SR_B1–B5 and B7), in the same order. They go in as in pretraining: reflectance with the GEE scale factors, clipped to 0–0.3 and divided by 0.3. SSL4EO-L mapped reflectance to 8 bits with the USGS scale factors and the GEE visualisation range 0–0.3, and torchgeo divides by 255; it is the same as `np.clip((x * 0.0000275 - 0.2) * 850, 0, 255) / 255` in 30_ReTraining_Torchgeo.
* **Other bands** (TEMP1 in kelvin / 300, the NDVI series and the wavelet bands, as in 09) get a new projection in the patch embedding. It starts at zero and is added to the pretrained projection of the six SR bands, so the model starts as the pretrained ViT and learns to use the other bands during training. `USE_TEMP1 = False` leaves TEMP1 out, so that `landsat` is the 6-band input of 30_ReTraining_Torchgeo.
* **Model:** 16 × 16-pixel patches of the 224 × 224 image (196 patches; 30 m pixels, as in the 224-pixel crops of SSL4EO-L), 12 transformer blocks of width 384, and one linear output on the class token, as torchgeo's `RegressionTask` builds it.
* **Training** (`VIT_MODE`, set in Settings or in the environment):
  * `finetune` (default): all layers learn. AdamW with learning rate 5e-5 for the pretrained weights and 1e-3 for the new projection and the output layer, weight decay 0.05 on the pretrained weight matrices, 5 epochs of linear warm-up and then cosine decay, stochastic depth 0.1, MSE loss, batch 64, 100 epochs, and the weights with the lowest validation loss. In 30_ReTraining_Torchgeo every weight had learning rate 1e-3, and fine-tuning ended with a higher validation MSE (about 243) than training the output layer alone (about 140 with `freeze_backbone=True`): a pretrained transformer needs a much smaller learning rate.
  * `frozen`: the pretrained weights do not change, as with `freeze_backbone=True`; only the new projection and the output layer learn (learning rate 1e-3, same schedule). Without other bands (`landsat` with `USE_TEMP1 = False`) this is a linear model on the pretrained features, which are then the same in every fold.
* **Clusters, folds and output** as in 09: by default all clusters of 03 (both periods), five models per row (test fold f, validation fold f − 1, training on the other three), and for every model `features.csv` with all sets:

| model | test fold | validation fold | training folds | Keras name |
|---|---|---|---|---|
| `test_E` | E | D | A, B, C | Fold12345 |
| `test_A` | A | E | B, C, D | Fold23451 |
| `test_B` | B | A | C, D, E | Fold34512 |
| `test_C` | C | B | D, E, A | Fold45123 |
| `test_D` | D | C | E, A, B | Fold51234 |

**Output,** in `landsat_vits16_runs/<sensors>_<mode>/<key>/` (by default `DMSP_VIIRS_finetune/`):

* `test_<fold>/features.csv`: the features of all sets of that model, one row for every cluster with complete data. In `landsat/test_E/features.csv`, for example, the clusters of folds A, B and C have `role` train, those of fold D val and those of fold E test. Columns as in 08 and 09: `cluster_id`, `country`, `year`, `rural`, `lat`, `lon`, `nl_sensor` (the period), `fold`, `role`, `iwi`, `iwi_pred` and `vit_feat_0` … `vit_feat_383` (08: `nl_feat_`, 09: `ls_feat_`, so that the files can be merged on `cluster_id`). For a second-stage model tested on fold f, use the files of model `test_f`.
* `features_oof.csv`: every cluster once, with the features and prediction of the model for which its fold was the test fold. It is written when the five models of the row are done.
* `test_<fold>/test_predictions.csv`, `result.json`, `history.csv` and `best.pt` (the weights), and the tables and plot of section 6 for all rows, next to ResNet-18 (09) and NL only (08).

**Running.** Check `WORK_DIR` and `TORCH_HOME` in Settings, and download the ViT-S/16 weights once (section 4). The band caches of 04 to 09 (`table1_cache/<sensor>_monthly_dmey_symmetric`) are used as they are, and a missing one is built first. A row's bands go into GPU memory when they need less than `GPU_DATA_GB`; otherwise every batch is read from the cache, which is slower (the log says which, and prints the time per epoch). Every model saves its state after each epoch, so a stopped job continues where it stopped.

**From random weights.** `TRAIN_FROM_SCRATCH=1` in the environment trains the `finetune` model from random weights: timm's initialisation (PyTorch's for the convolutions of the patch embedding, so the projection of the other bands no longer starts at zero), and the weights file is not read. The transformer then learns with `LR_SCRATCH` = 5e-5 in place of `LR_PRETRAINED`, close to DeiT's rule for training from scratch (5e-4 × batch / 512 = 6e-5); the warm-up, cosine decay, weight decay and stochastic depth are those of DeiT too. If it underfits, try 1e-4 to 2e-4, not 1e-3. Give these runs their own `RUN_ROOT`: a run folder holds one kind of run, and the notebook stops when it finds the other kind. The comparison with 09 and 08 still reads `RESNET_RUNS` and `NL_RUNS`.

> **Numbering.** This notebook was written before the notebooks were renumbered for this repository. Apart from the title, its text, comments and plot labels use the earlier numbers: 05 = `04_Table2_DHS`, 07 = `05_Table2_ResNet18`, 08 = `06_NL_ResNet18_Features`, 09 = `07_Landsat_ResNet18_Features`, 10 = `08_Landsat_ViTS16_Features`, 11 = `09_Ridge_NL_Landsat` and 12 = `10_Ridge_NL_ViT`; 01 to 03 are unchanged. The earlier 04 and 06 belonged to the simulated dataset of the preliminary study and are not included. Folder names such as `ridge_runs/nl08_landsat09` keep the earlier numbers.

# Import libraries

In [ ]:
import os
import json
import math
import time
import uuid
import datetime
import multiprocessing

# Same as in 01-08: the GPU image sets PROJ_LIB to a folder without proj.db, which breaks rasterio's CRS handling.
for _var in ('PROJ_LIB', 'PROJ_DATA'):
    if os.environ.get(_var) and not os.path.exists(os.path.join(os.environ[_var], 'proj.db')):
        os.environ.pop(_var)

import numpy as np
import pandas as pd
import pywt
import timm
import rasterio
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
from IPython.display import display
try:
    from tqdm.auto import tqdm
except ImportError:                      # progress bars are optional
    def tqdm(it, **kw):
        return it

# Settings

In [ ]:
# ---- model ----------------------------------------------------------------------------------------
VIT_MODE  = os.environ.get('VIT_MODE', 'finetune')   # 'finetune': all layers learn; 'frozen': the pretrained weights
                                                     # stay fixed (freeze_backbone=True of 30_ReTraining_Torchgeo)
USE_TEMP1 = True           # TEMP1 in every row, as in 09 (False: Landsat = the 6 SR bands of 30_ReTraining_Torchgeo)

# ---- input / output ------------------------------------------------------------------------
WORK_DIR   = '/path/to/data/'
FOLDS_CSV  = os.path.join(WORK_DIR, 'train_clusters_folds.csv')   # written by 03_Spatial_Folds
BAND_CACHE = os.path.join(WORK_DIR, 'table1_cache')               # Landsat, NDVI and wavelet bands, ~2.5 MB per cluster
                                                                  # (shared with 04-09)
TORCH_HOME = '/path/to/torch_home'   # ViT-S/16 weights (hub/checkpoints)

# ---- data (as in 04-09, so that the band caches are shared) --------------------------------------------------
TARGET       = 'iwi'                # International Wealth Index of the cluster (0-100), a column of FOLDS_CSV
NL_SENSORS   = ['DMSP', 'VIIRS']    # the clusters of these periods of 01 (the NL image itself is not an input): DMSP =
                                    # surveys up to 2013, VIIRS = from 2014. Both: all clusters, as in 08 and 09
NDVI_SERIES  = 'monthly'            # 'monthly': the 23 MODIS composites averaged to 12 months, the paper's 12 values;
                                    # '16day': all 23 composites as the series
WAVELET      = 'dmey'               # discrete Meyer, filter length 62, as in the paper
WAVELET_MODE = 'symmetric'          # pywt's default signal extension
NODATA       = -999999.0            # no-data value of the 01 files; clusters with any no-data pixel are left out

# ---- normalisation of the inputs (fixed numbers) ----------------------------------------------------------------
# The 6 SR bands as in SSL4EO-L: reflectance (DN * 0.0000275 - 0.2, the GEE scale factors) clipped to 0-SR_MAX and
# divided by SR_MAX (SSL4EO-L: 0-0.3 -> 0-255 uint8, torchgeo: / 255). TEMP1: DN * 0.00341802 + 149.0 = kelvin, divided
# by TEMP1_DIVISOR. NDVI and wavelet bands as they are (as in 04-09).
SR_MAX        = 0.3        # the GEE visualisation range of Landsat Collection 2 SR, used by SSL4EO-L
TEMP1_DIVISOR = 300.0      # kelvin / 300, as in 04-09

# ---- training -----------------------------------------------------------------------------------------------------
FOLDS         = ['A', 'B', 'C', 'D', 'E']
EPOCHS        = 100
BATCH_SIZE    = 64
LR_PRETRAINED = 5e-5       # AdamW learning rate of the pretrained weights (finetune)
LR_NEW        = 1e-3       # ... of the new ones: the projection of the other bands and the output layer
LR_SCRATCH    = 5e-5       # LR_PRETRAINED with TRAIN_FROM_SCRATCH (DeiT from scratch: 5e-4 x 64 / 512 = 6e-5)
WEIGHT_DECAY  = 0.05       # on the pretrained weight matrices (not biases, norms, position embedding, class token)
WARMUP_EPOCHS = 5          # linear warm-up of the learning rates, then cosine decay to 0 at EPOCHS
DROP_PATH     = 0.1        # stochastic depth of the transformer blocks (finetune)
SEED          = 3127
AMP           = True       # bfloat16 autocast on the GPU for training; the features are computed in float32
GPU_DATA_GB   = 60         # a row's bands go into GPU memory when they need less than this (60 000 clusters: 42 GB
                           # for 7 bands, 114 GB for 19), otherwise every batch is read from the caches (slower)
N_WORKERS     = len(os.sched_getaffinity(0))   # processes that build a band cache if it is missing

# ---- output -------------------------------------------------------------------------------------------------
FEATURE_PREFIX = 'vit_feat_'  # columns vit_feat_0 ... vit_feat_383 (08: nl_feat_, 09: ls_feat_)
PERIODS = '_'.join(NL_SENSORS)
RUN_ROOT = os.path.join(WORK_DIR, 'landsat_vits16_runs',                 # one folder per row
                        f'{PERIODS}_{VIT_MODE}' + ('' if USE_TEMP1 else '_noTEMP1'))
RESNET_RUNS = os.path.join(WORK_DIR, 'landsat_resnet18_runs', f'{PERIODS}_imagenet')   # 09, for the comparison
NL_RUNS     = os.path.join(WORK_DIR, 'nl_resnet18_runs', f'{PERIODS}_imagenet')        # 08, for the comparison

# ---- what to run: rows ('all', 'none' = only the caches and the results, or keys of ROWS) and test folds ('all' or
# folds), comma-separated. These can also be set in the environment.
RUN_ROWS  = os.environ.get('VITFEAT_ROWS', 'all')
RUN_FOLDS = os.environ.get('VITFEAT_FOLDS', 'all')
RUN_FOLDS = FOLDS if RUN_FOLDS == 'all' else [f.strip() for f in RUN_FOLDS.split(',')]
assert set(RUN_FOLDS) <= set(FOLDS), f'unknown test folds: {RUN_FOLDS}'
assert set(NL_SENSORS) <= {'DMSP', 'VIIRS'}, NL_SENSORS
assert VIT_MODE in ('finetune', 'frozen'), f'VIT_MODE must be finetune or frozen, not {VIT_MODE}'
KERAS_NAME = {'E': 'Fold12345', 'A': 'Fold23451', 'B': 'Fold34512', 'C': 'Fold45123', 'D': 'Fold51234'}  # 27_Train_NL

# ---- TRAIN_FROM_SCRATCH=1 in the environment: the finetune model from random weights
# (timm's initialisation; the weights file is not read), with LR_SCRATCH in place of LR_PRETRAINED. Give these runs
# their own RUN_ROOT: a run folder holds one kind of run (check_start() stops the notebook otherwise).
_scratch = os.environ.get('TRAIN_FROM_SCRATCH', '0').strip()
assert _scratch in ('', '0', '1'), f'TRAIN_FROM_SCRATCH must be 0 or 1, not {_scratch!r}'
TRAIN_FROM_SCRATCH = _scratch == '1'
if TRAIN_FROM_SCRATCH:
    assert VIT_MODE == 'finetune', 'TRAIN_FROM_SCRATCH=1 needs VIT_MODE=finetune (frozen keeps the weights fixed)'
    LR_PRETRAINED = LR_SCRATCH
VIT_NAME = 'ViT-S/16 (random start)' if TRAIN_FROM_SCRATCH else f'ViT-S/16 (SSL4EO-L MoCo, {VIT_MODE})'

os.environ['TORCH_HOME'] = TORCH_HOME
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
torch.backends.cudnn.benchmark = True
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True
print('device:', DEVICE, torch.cuda.get_device_name(0) if DEVICE.type == 'cuda' else '(no GPU: only for code tests)')
print('torch', torch.__version__, '| timm', timm.__version__, '| pywt', pywt.__version__, '| cache workers:', N_WORKERS)
print('ViT-S/16', VIT_MODE, '| from random weights:', TRAIN_FROM_SCRATCH, '| TEMP1:', USE_TEMP1, '| periods:',
      NL_SENSORS, '| runs ->', RUN_ROOT)

# 1. The rows

Every row starts with the six SR bands of the pretrained model and TEMP1 (unless `USE_TEMP1 = False`), the Landsat bands of 09. The 6 wavelet bands are in the order of the band cache (and of bands 12–17 of the paper's TF records): LFC minimum, maximum, mean, then HFC minimum, maximum, mean.

In [ ]:
LANDSAT = ['BLUE', 'GREEN', 'RED', 'NIR', 'SWIR1', 'SWIR2', 'TEMP1']
N_NDVI = 12 if NDVI_SERIES == 'monthly' else 23
NDVI = [f'NDVI_{k:02d}' for k in range(1, N_NDVI + 1)]
WAVELET_BANDS = ['LFC_min', 'LFC_max', 'LFC_mean', 'HFC_min', 'HFC_max', 'HFC_mean']
CHANNELS = LANDSAT + NDVI + WAVELET_BANDS          # the bands of the band cache of 04-09, in this order
SR_BANDS = LANDSAT[:6]          # the bands of the pretrained weights (ETM+ SR_B1-B5, B7), in their order
BASE = SR_BANDS + (['TEMP1'] if USE_TEMP1 else [])

# key: (input, bands)
ROWS = {
    'landsat':          ('Landsat images', BASE),
    'landsat_ndvi':     ('Landsat images and temporal MODIS NDVI', BASE + NDVI),
    'landsat_wavelet6': ('Landsat images and 6 statistical wavelet-based features', BASE + WAVELET_BANDS),
    'landsat_hfc_max':  ('Landsat images and max values of HFC', BASE + ['HFC_max']),
    'landsat_hfc_mean': ('Landsat images and mean values of HFC', BASE + ['HFC_mean']),
    'landsat_hfc_min':  ('Landsat images and min values of HFC', BASE + ['HFC_min']),
    'landsat_lfc_max':  ('Landsat images and max values of LFC', BASE + ['LFC_max']),
    'landsat_lfc_mean': ('Landsat images and mean values of LFC', BASE + ['LFC_mean']),
    'landsat_lfc_min':  ('Landsat images and min values of LFC', BASE + ['LFC_min']),
}

if RUN_ROWS == 'all':
    todo = list(ROWS)
elif RUN_ROWS == 'none':
    todo = []
else:
    todo = [k.strip() for k in RUN_ROWS.split(',') if k.strip()]
bad = [k for k in todo if k not in ROWS]
assert not bad, f'unknown rows: {bad}'
print('rows to run:', todo, '| test folds:', RUN_FOLDS)
pd.DataFrame([(k, d, len(b), ', '.join(b[len(BASE):]) or '-') for k, (d, b) in ROWS.items()],
             columns=['key', 'input', 'bands', 'bands added to ' + ' + '.join(['6 SR'] + BASE[6:])])

# 2. Clusters

All clusters of `train_clusters_folds.csv` of the periods in `NL_SENSORS`. For every period they are in the order of 04 to 09 (the clusters of that NL sensor, sorted by `cluster_id`), so that the band caches are shared. They keep the fold of 03.

In [ ]:
table = pd.read_csv(FOLDS_CSV)
if 'nl_sensor' not in table:                       # 01 uses DMSP up to 2013 and VIIRS from 2014
    table['nl_sensor'] = np.where(table['year'] >= 2014, 'VIIRS', 'DMSP')
if 'folder' not in table:
    table['folder'] = [os.path.join(WORK_DIR, 'Dataset', str(c).replace('/', '_')) for c in table['cluster_id']]
assert TARGET in table, f'{FOLDS_CSV} has no column {TARGET}'

# the clusters of every period in the order of 04-07, then one table: part = period, part_row = row in its band cache
PARTS = {s: table[table['nl_sensor'] == s].sort_values('cluster_id').reset_index(drop=True) for s in NL_SENSORS}
clusters = pd.concat([p.assign(part=s, part_row=np.arange(len(p))) for s, p in PARTS.items()], ignore_index=True)
assert clusters['cluster_id'].is_unique, 'a cluster_id appears twice in FOLDS_CSV'
has_target = clusters[TARGET].notna().to_numpy()

left = table.loc[~table['nl_sensor'].isin(NL_SENSORS), 'nl_sensor']
print(f'{len(clusters)} clusters', clusters['nl_sensor'].value_counts().to_dict(),
      f'| {has_target.sum()} with {TARGET} | {clusters["country"].nunique()} countries, '
      f'years {clusters["year"].min()}-{clusters["year"].max()}'
      + (f' | left out (other period): {left.value_counts().to_dict()}' if len(left) else ''))
print(f'{TARGET}:', clusters.loc[has_target, TARGET].describe().round(2).to_dict())
if clusters['fold'].isna().any():
    print(f"{clusters['fold'].isna().sum()} clusters have no fold: they get features but no model is trained or "
          'tested on them')
print()
print(pd.crosstab(clusters['nl_sensor'], clusters['fold'], margins=True).to_string())

# 3. Band caches

Every cluster's bands are computed once from its three GeoTIFFs of 01 and stored as float16 in `table1_cache/<sensor>_monthly_dmey_symmetric/`, one folder per period, the folders and format of 04 to 09 (about 2.5 MB per cluster): the 7 Landsat bands with the GEE scale factors, surface reflectance (DN × 0.0000275 − 0.2) and TEMP1 in kelvin / 300 ((DN × 0.00341802 + 149) / 300); the NDVI series (for `monthly`, each MODIS composite counts for the month in which it starts, and the composites of a month are averaged); and the 6 wavelet bands of that series. A cluster with a no-data pixel in any of its three files (the NL image included) is left out, as in 04 to 09. The scaling of the SR bands for the pretrained model and of TEMP1 is applied when the bands are loaded (`normalise()`).

A complete cache is used as it is, a missing one is built, and a job that finds a cache being built by another job (of 04 to 10) waits for it. The table at the end shows the training values of every band for a sample of clusters.

In [ ]:
PATCH = 224
BAND_DIRS = {s: os.path.join(BAND_CACHE, f'{s}_{NDVI_SERIES}_{WAVELET}_{WAVELET_MODE}') for s in NL_SENSORS}


def month_of_composites(year):
    """Calendar month (1-12) in which each of the 23 MOD13Q1 composites of a year starts (DOY 1, 17, ..., 353)."""
    jan1 = datetime.date(int(year), 1, 1)
    return np.array([(jan1 + datetime.timedelta(days=16 * k)).month for k in range(23)])


def cluster_bands(folder, year):
    """The bands of the band cache for one cluster (float32, CHANNELS order) and its mean NL, or None if a file has
    no-data pixels. TEMP1 is kelvin / 300 here; normalise() applies TEMP1_DIVISOR."""
    with rasterio.open(os.path.join(folder, 'Landsat.tif')) as src:
        ls = src.read().astype(np.float32)
    with rasterio.open(os.path.join(folder, 'MODIS_NDVI.tif')) as src:
        nd = src.read().astype(np.float32)
    with rasterio.open(os.path.join(folder, 'NL.tif')) as src:
        nl = src.read(1).astype(np.float64)
    if (ls == NODATA).any() or (nd == NODATA).any() or (nl == NODATA).any():
        return None
    ls[:6] = ls[:6] * 0.0000275 - 0.2                     # surface reflectance
    ls[6] = (ls[6] * 0.00341802 + 149.0) / 300.0          # kelvin / 300
    if NDVI_SERIES == 'monthly':
        month = month_of_composites(year)
        nd = np.stack([nd[month == m].mean(axis=0) for m in range(1, 13)])
    lfc, hfc = pywt.dwt(nd, WAVELET, mode=WAVELET_MODE, axis=0)    # single-level DWT along time, every pixel
    wav = np.stack([lfc.min(0), lfc.max(0), lfc.mean(0), hfc.min(0), hfc.max(0), hfc.mean(0)])
    return np.concatenate([ls, nd, wav]).astype(np.float32), float(nl.mean())


def _cache_one(task):
    """Compute one cluster and write it at its place in the .part file. Returns the mean NL and the band sums."""
    i, folder, year, offset, part = task
    try:
        out = cluster_bands(folder, year)
    except Exception as e:
        return i, np.nan, None, f'{type(e).__name__}: {e}'[:200]
    if out is None:
        return i, np.nan, None, 'no-data pixels'
    x, nl_mean = out
    fd = os.open(part, os.O_WRONLY)
    try:
        os.pwrite(fd, x.astype(np.float16).tobytes(), offset)
    finally:
        os.close(fd)
    x64 = x.astype(np.float64)
    return i, nl_mean, np.stack([x64.sum(axis=(1, 2)), (x64 ** 2).sum(axis=(1, 2))]), ''


def build_cache(cache_dir, table):
    """features.npy, band_stats.csv and index.csv of the band cache of the clusters in table (the format of 04-07)."""
    t0 = time.time()
    features = os.path.join(cache_dir, 'features.npy')
    shape = (len(table), len(CHANNELS), PATCH, PATCH)
    arr = np.lib.format.open_memmap(features + '.part', mode='w+', dtype=np.float16, shape=shape)
    header, per_cluster = arr.offset, int(np.prod(shape[1:])) * 2
    del arr
    tasks = [(i, r.folder, int(r.year), header + i * per_cluster, features + '.part')
             for i, r in enumerate(table.itertuples())]
    nl_mean = np.full(len(table), np.nan)
    sums = np.zeros((2, len(CHANNELS)))
    error = [''] * len(table)
    with multiprocessing.get_context('fork').Pool(N_WORKERS) as pool:
        for i, t, s, err in tqdm(pool.imap_unordered(_cache_one, tasks, chunksize=4), total=len(tasks), desc='bands'):
            nl_mean[i], error[i] = t, err
            if s is not None:
                sums += s
    ok = np.isfinite(nl_mean)
    n_pix = ok.sum() * PATCH * PATCH
    mean = sums[0] / n_pix
    std = np.sqrt(np.maximum(sums[1] / n_pix - mean ** 2, 1e-12))
    pd.DataFrame({'band': CHANNELS, 'mean': mean, 'std': std}).to_csv(os.path.join(cache_dir, 'band_stats.csv'),
                                                                      index=False)
    os.replace(features + '.part', features)
    index = table[['cluster_id', 'country', 'year', 'lat', 'lon']].copy()
    index['nl_mean'], index['ok'], index['error'] = nl_mean, ok, error
    index.to_csv(os.path.join(cache_dir, 'index.csv'), index=False)    # the cache counts as complete once this exists
    print(f'bands cache: {ok.sum()} of {len(ok)} clusters in {time.time() - t0:.0f} s -> {features}')


def cache_is_complete(cache_dir, table):
    index, features = os.path.join(cache_dir, 'index.csv'), os.path.join(cache_dir, 'features.npy')
    if not (os.path.exists(index) and os.path.exists(features)):
        return False
    return pd.read_csv(index)['cluster_id'].tolist() == table['cluster_id'].tolist()


def ensure_cache(cache_dir, table):
    """Build a band cache folder unless it is complete. Waits when another job is building it (same lock in 04-10)."""
    os.makedirs(cache_dir, exist_ok=True)
    lock = os.path.join(cache_dir, 'building.lock')
    if cache_is_complete(cache_dir, table):
        print('band cache found:', cache_dir)
        return
    try:
        os.mkdir(lock)
    except FileExistsError:
        print(f'another job is building {cache_dir} (if none is running, delete {lock}); waiting ...')
        t_wait = time.time()
        while os.path.exists(lock) and time.time() - t_wait < 4 * 3600:
            time.sleep(30)
        assert cache_is_complete(cache_dir, table), 'the cache is still not complete: start the notebook again'
        return
    try:
        if not cache_is_complete(cache_dir, table):     # another job may have finished it just now
            build_cache(cache_dir, table)
    finally:
        os.rmdir(lock)


def read_bands(pos, ch):
    """Cached float16 bands ch (positions in CHANNELS) of the clusters at the rows pos of `clusters`, shape
    (len(pos), len(ch), 224, 224)."""
    pos, ch = np.asarray(pos), np.asarray(ch)
    out = np.empty((len(pos), len(ch), PATCH, PATCH), np.float16)
    part, prow = clusters['part'].to_numpy()[pos], clusters['part_row'].to_numpy()[pos]
    for s in np.unique(part):
        k = np.flatnonzero(part == s)
        k = k[np.argsort(prow[k])]                      # read in file order
        out[k] = FEAT[s][prow[k][:, None], ch[None, :]]
    return out


TEMP_FROM_CACHE = 300.0 / TEMP1_DIVISOR    # the band cache keeps TEMP1 in kelvin / 300


def normalise(x, bands):
    """Training values of cached bands, in place (float32 tensor (n, bands, 224, 224)): the SR bands as in SSL4EO-L
    (reflectance clipped to 0-SR_MAX, divided by SR_MAX), TEMP1 in kelvin / TEMP1_DIVISOR, NDVI and wavelet bands as
    they are."""
    for k, b in enumerate(bands):
        if b in SR_BANDS:
            x[:, k] = (x[:, k] / SR_MAX).clamp_(0, 1)
        elif b == 'TEMP1' and TEMP_FROM_CACHE != 1.0:
            x[:, k] *= TEMP_FROM_CACHE
    return x


for s in NL_SENSORS:
    ensure_cache(BAND_DIRS[s], PARTS[s])
BAND_INDEX = {s: pd.read_csv(os.path.join(BAND_DIRS[s], 'index.csv')) for s in NL_SENSORS}
FEAT = {s: np.load(os.path.join(BAND_DIRS[s], 'features.npy'), mmap_mode='r') for s in NL_SENSORS}
for s in NL_SENSORS:
    assert FEAT[s].shape == (len(PARTS[s]), len(CHANNELS), PATCH, PATCH), (s, FEAT[s].shape)
    print(f'{s} period: band cache {FEAT[s].shape}, {FEAT[s].nbytes / 1e9:.1f} GB -> {BAND_DIRS[s]}')
complete = np.concatenate([BAND_INDEX[s]['ok'].to_numpy(bool) for s in NL_SENSORS])
errors = pd.Series(np.concatenate([BAND_INDEX[s]['error'].fillna('').to_numpy(str) for s in NL_SENSORS]))
print(f'clusters with complete data: {complete.sum()} of {len(complete)}; left out:',
      errors[~complete].str.split(':').str[0].value_counts().to_dict())
assert complete.any(), 'no cluster has complete data'

# training values of every band for a sample of clusters of every period
rng = np.random.default_rng(0)
sample = np.sort(np.concatenate([rng.choice(c, size=min(16, len(c)), replace=False) for c in
                                 (np.flatnonzero(complete & (clusters['part'] == s).to_numpy()) for s in NL_SENSORS)
                                 if len(c)]))
x = normalise(torch.from_numpy(read_bands(sample, np.arange(len(CHANNELS))).astype(np.float32)), CHANNELS)
print(f'\ntraining values of the input bands, {len(sample)} clusters:',
      clusters['part'].to_numpy()[sample].tolist().count('DMSP'), 'DMSP and',
      clusters['part'].to_numpy()[sample].tolist().count('VIIRS'), 'VIIRS period')
pd.DataFrame({'min': x.amin(dim=(0, 2, 3)).numpy(), 'mean': x.mean(dim=(0, 2, 3)).numpy(),
              'max': x.amax(dim=(0, 2, 3)).numpy()}, index=CHANNELS).T.round(3)

In [ ]:
# check the wealthiest cluster of every period against a fresh computation from its files, then show its bands
usable = complete & has_target
fig, ax = plt.subplots(len(NL_SENSORS), 4, figsize=(15, 3.9 * len(NL_SENSORS)), constrained_layout=True, squeeze=False)
for r, s in enumerate(NL_SENSORS):
    cand = np.flatnonzero(usable & (clusters['part'] == s).to_numpy())
    if not len(cand):
        for a in ax[r]:
            a.axis('off')
        continue
    i = int(cand[np.argmax(clusters[TARGET].to_numpy()[cand])])
    x_new, nl_mean_new = cluster_bands(clusters.at[i, 'folder'], clusters.at[i, 'year'])
    x = read_bands([i], np.arange(len(CHANNELS)))[0].astype(np.float32)
    assert np.allclose(x, x_new, rtol=2e-3, atol=2e-3)
    assert np.isclose(nl_mean_new, BAND_INDEX[s].at[clusters.at[i, 'part_row'], 'nl_mean'])
    print(f'{s} period: cache matches the files for', clusters.at[i, 'cluster_id'])
    ax[r, 0].imshow(np.moveaxis(np.clip(x[[2, 1, 0]] / 0.3, 0, 1), 0, -1))
    ax[r, 0].set_title(f"{clusters.at[i, 'cluster_id']} ({clusters.at[i, 'year']}), {TARGET} "
                       f"{clusters.at[i, TARGET]:.1f}\nLandsat RGB", fontsize=9)
    nd = x[len(LANDSAT):len(LANDSAT) + N_NDVI]
    ax[r, 1].plot(range(1, N_NDVI + 1), nd.reshape(N_NDVI, -1).mean(axis=1), 'o-', color='#2a78d6')
    ax[r, 1].set_title('NDVI series, patch mean', fontsize=9)
    ax[r, 1].set_xlabel('month' if NDVI_SERIES == 'monthly' else '16-day composite')
    for c, name in ((2, 'LFC_mean'), (3, 'HFC_mean')):
        fig.colorbar(ax[r, c].imshow(x[CHANNELS.index(name)], cmap='viridis'), ax=ax[r, c], shrink=0.8)
        ax[r, c].set_title(name, fontsize=9)
    for c in (0, 2, 3):
        ax[r, c].set_xticks([])
        ax[r, c].set_yticks([])
fig.suptitle('The wealthiest cluster of every period, bands as cached')
plt.show()

# 4. Model and training

`LandsatViT` is timm's `vit_small_patch16_224` with the weights of `ViTSmall16_Weights.LANDSAT_ETM_SR_MOCO` (loaded as torchgeo loads them: the file may lack the final norm and the head), a `PatchProjection` for the bands of the row, and `head`, a linear layer that runs in float32. `features()` gives the class token after the final layer norm (384 values), the input of `head`. The cell checks that a model with other bands starts as the pretrained ViT on the six SR bands. With `TRAIN_FROM_SCRATCH` (Settings) the weights file is not read: the model keeps timm's random initialisation, the projection of the other bands PyTorch's (not zero), every layer learns, and the transformer gets `LR_SCRATCH`. `check_start()` stops the notebook when a run folder holds a run of the other kind (`from_scratch` in `result.json` and the checkpoints).

A run trains the model of one row and test fold, as in 09. After every epoch it writes the history (with the learning rates), the best weights (`best.pt`, lowest validation loss; in `frozen` mode only the new parameters) and its full state (`last.pt`). Then it loads `best.pt`, computes the features and the prediction of every cluster in float32 and writes `features.csv` (all sets), `test_predictions.csv` and `result.json`, and deletes `last.pt`, which is only needed to continue a stopped run. A run with `result.json` and `features.csv` is skipped. A run with only a `last.pt` continues from that epoch. A finished run whose `features.csv` was deleted writes it again from `best.pt`.

If the machine has no internet access, download the weights once beforehand (torch then checks the file against the hash in its name):

```bash
mkdir -p /path/to/torch_home/hub/checkpoints
wget -O /path/to/torch_home/hub/checkpoints/vits16_landsat_etm_sr_moco-eaa4674e.pth \
     https://hf.co/torchgeo/ssl4eo_landsat/resolve/1c88bb51b6e17a21dde5230738fa38b74bd74f76/vits16_landsat_etm_sr_moco-eaa4674e.pth
```

In [ ]:
WEIGHTS_URL = ('https://hf.co/torchgeo/ssl4eo_landsat/resolve/1c88bb51b6e17a21dde5230738fa38b74bd74f76/'
               'vits16_landsat_etm_sr_moco-eaa4674e.pth')     # ViTSmall16_Weights.LANDSAT_ETM_SR_MOCO of torchgeo 0.10
try:                                                           # the URL of the installed torchgeo, if there is one
    from torchgeo.models import ViTSmall16_Weights
    WEIGHTS_URL = ViTSmall16_Weights.LANDSAT_ETM_SR_MOCO.url
except Exception:                                              # torchgeo missing or failing: the URL above
    pass
WEIGHTS_FILE = os.path.join(TORCH_HOME, 'hub', 'checkpoints', os.path.basename(WEIGHTS_URL))
VIT_STATE = None                                               # TRAIN_FROM_SCRATCH: random weights, no file needed
if not TRAIN_FROM_SCRATCH:
    try:
        VIT_STATE = torch.hub.load_state_dict_from_url(WEIGHTS_URL, map_location='cpu', progress=False,
                                                        check_hash=True, weights_only=True)
    except Exception as e:
        raise RuntimeError(f'the ViT-S/16 weights could not be loaded ({type(e).__name__}: {e}). Download them '
                           f'first:\n  mkdir -p {os.path.dirname(WEIGHTS_FILE)}\n'
                           f'  wget -O {WEIGHTS_FILE} {WEIGHTS_URL}') from e
OPTIONAL_KEYS = {'norm.weight', 'norm.bias', 'head.weight', 'head.bias'}    # may be missing in the file (torchgeo)
if VIT_STATE is not None:
    print('ViT-S/16 weights:', WEIGHTS_FILE, '| patch embedding', tuple(VIT_STATE['patch_embed.proj.weight'].shape),
          '| final norm in the file:', 'norm.weight' in VIT_STATE)
else:
    print('ViT-S/16: random start (TRAIN_FROM_SCRATCH), the weights file is not read')


class PatchProjection(nn.Module):
    """The patch embedding of ViT-S/16 for the bands of a row (the six SR bands first): the pretrained projection of
    the SR bands plus a new projection of the other bands, which starts at zero (TRAIN_FROM_SCRATCH: both start
    with PyTorch's random initialisation of a convolution)."""

    def __init__(self, pretrained, n_extra):
        super().__init__()
        self.sr = pretrained                                   # Conv2d(6, 384, 16, stride 16), with bias
        self.extra = None
        if n_extra:
            self.extra = nn.Conv2d(n_extra, pretrained.out_channels, kernel_size=pretrained.kernel_size,
                                   stride=pretrained.stride, bias=False)
            if not TRAIN_FROM_SCRATCH:                         # the model then starts as the pretrained ViT
                nn.init.zeros_(self.extra.weight)

    def forward(self, x):
        out = self.sr(x[:, :len(SR_BANDS)])
        if self.extra is not None:
            out = out + self.extra(x[:, len(SR_BANDS):])
        return out


def is_new(name):
    """Parameters that are not in the pretrained weights: the output layer and the projection of the other bands."""
    return name.startswith('head.') or '.proj.extra.' in name


class LandsatViT(nn.Module):
    """ViT-S/16 with the SSL4EO-L MoCo weights for the bands of a row. features(): the class token after the final
    layer norm (384 values, the input of the output layer); forward(): the predicted IWI. TRAIN_FROM_SCRATCH: timm's
    random initialisation in place of the SSL4EO-L weights."""

    def __init__(self, bands):
        super().__init__()
        assert list(bands[:len(SR_BANDS)]) == SR_BANDS, 'the six SR bands come first'
        net = timm.create_model('vit_small_patch16_224', in_chans=len(SR_BANDS), num_classes=0,
                                drop_path_rate=DROP_PATH if VIT_MODE == 'finetune' else 0.0)
        if VIT_STATE is not None:                              # else (TRAIN_FROM_SCRATCH) timm's random weights
            missing, unexpected = net.load_state_dict(VIT_STATE, strict=False)
            assert set(missing) <= OPTIONAL_KEYS and set(unexpected) <= OPTIONAL_KEYS, (missing, unexpected)
        net.patch_embed.proj = PatchProjection(net.patch_embed.proj, len(bands) - len(SR_BANDS))
        self.backbone = net                                    # num_classes=0: returns the class token, no head
        self.head = nn.Linear(net.num_features, 1)
        nn.init.xavier_uniform_(self.head.weight)              # as in 08 and 09; the bias is set to the train mean
        nn.init.zeros_(self.head.bias)

    def features(self, x):
        return self.backbone(x)

    def forward(self, x):
        f = self.features(x)
        with torch.autocast(device_type=f.device.type, enabled=False):    # output layer in float32
            return self.head(f.float()).squeeze(1)

    def train(self, mode=True):
        super().train(mode)
        if VIT_MODE == 'frozen':
            self.backbone.eval()                               # no stochastic depth in the frozen blocks
        return self

    def learned_state(self):
        """What a checkpoint keeps: all weights (finetune) or the new ones (frozen)."""
        state = self.state_dict()
        return state if VIT_MODE == 'finetune' else {k: v for k, v in state.items() if is_new(k)}

    def load_learned_state(self, state):
        missing, unexpected = self.load_state_dict(state, strict=False)
        assert not unexpected and (VIT_MODE == 'frozen' or not missing), (missing, unexpected)


def make_optimizer(model):
    """AdamW: LR_PRETRAINED for the pretrained weights, LR_NEW for the new ones; weight decay on the pretrained weight
    matrices only. frozen: the pretrained weights do not learn. TRAIN_FROM_SCRATCH: the same groups, with random
    weights and LR_PRETRAINED = LR_SCRATCH."""
    groups = {'decay': [], 'no_decay': [], 'new': []}
    for name, p in model.named_parameters():
        if is_new(name):
            groups['new'].append(p)
        elif VIT_MODE == 'frozen':
            p.requires_grad_(False)
        elif p.ndim >= 2 and not name.endswith(('pos_embed', 'cls_token')):
            groups['decay'].append(p)
        else:
            groups['no_decay'].append(p)
    spec = [{'params': groups['decay'], 'lr': LR_PRETRAINED, 'weight_decay': WEIGHT_DECAY, 'name': 'pretrained'},
            {'params': groups['no_decay'], 'lr': LR_PRETRAINED, 'weight_decay': 0.0, 'name': 'pretrained_no_decay'},
            {'params': groups['new'], 'lr': LR_NEW, 'weight_decay': 0.0, 'name': 'new'}]
    spec = [g for g in spec if g['params']]
    for g in spec:
        g['base_lr'] = g['lr']
    return torch.optim.AdamW(spec)


def lr_factor(t):
    """Learning-rate factor at epoch t (fractional): linear warm-up over WARMUP_EPOCHS, then cosine decay to 0."""
    if t < WARMUP_EPOCHS:
        return t / WARMUP_EPOCHS
    return 0.5 * (1 + math.cos(math.pi * min(1.0, (t - WARMUP_EPOCHS) / max(EPOCHS - WARMUP_EPOCHS, 1))))


class Bands:
    """The bands of one row for the clusters at the rows `rows` of `clusters`, from the band caches, with the fixed
    normalisation of normalise(). The cached values go into DEVICE memory when they take less than GPU_DATA_GB,
    otherwise every batch is read from the caches (slower)."""

    def __init__(self, bands, rows):
        self.bands = list(bands)
        self.n_bands = len(self.bands)
        self.rows = np.asarray(rows)
        self.ch = np.array([CHANNELS.index(b) for b in self.bands])
        gb = len(self.rows) * self.n_bands * PATCH * PATCH * 2 / 1e9
        self.x = None
        if gb < GPU_DATA_GB:
            self.x = torch.empty((len(self.rows), self.n_bands, PATCH, PATCH), dtype=torch.float16, device=DEVICE)
            for s in tqdm(range(0, len(self.rows), 256), desc=f'loading {self.n_bands} bands, {gb:.1f} GB',
                          leave=False):
                self.x[s:s + 256] = torch.from_numpy(read_bands(self.rows[s:s + 256], self.ch)).to(DEVICE)
        print(f'{self.n_bands} bands of {len(self.rows)} clusters, {gb:.1f} GB: ' + (
              'in GPU memory' if self.x is not None else
              f'more than GPU_DATA_GB = {GPU_DATA_GB}, so every batch is read from the caches'))

    def batch(self, idx):
        """Normalised float32 bands of the clusters at positions idx (a 1-D tensor)."""
        if self.x is not None:
            x = self.x[idx.to(self.x.device)]
        else:
            x = torch.from_numpy(read_bands(self.rows[idx.cpu().numpy()], self.ch)).to(DEVICE, non_blocking=True)
        x = normalise(x.float(), self.bands)
        return x.contiguous(memory_format=torch.channels_last)


def r2_score(y, p):
    y, p = np.asarray(y, np.float64), np.asarray(p, np.float64)
    return float(1 - ((y - p) ** 2).sum() / ((y - y.mean()) ** 2).sum())


def autocast():
    return torch.autocast(device_type=DEVICE.type, dtype=torch.bfloat16, enabled=AMP and DEVICE.type == 'cuda')


def predict(model, data, idx, batch=256):
    model.eval()
    out = []
    with torch.no_grad(), autocast():
        for s in range(0, len(idx), batch):
            out.append(model(data.batch(idx[s:s + batch])).float())
    return torch.cat(out)


def save(obj, path):
    torch.save(obj, path + '.tmp')
    os.replace(path + '.tmp', path)


def check_start(from_scratch, where):
    """A run folder holds one kind of run: stop instead of mixing runs from pretrained and from random weights
    (TRAIN_FROM_SCRATCH). Runs written before 2026-11-09 have no 'from_scratch' and started from pretrained weights."""
    if bool(from_scratch) != TRAIN_FROM_SCRATCH:
        kind = {True: 'from random weights', False: 'from the pretrained weights'}
        raise RuntimeError(f'{where} holds a run {kind[bool(from_scratch)]}, but this run starts '
                           f'{kind[TRAIN_FROM_SCRATCH]}: give it another RUN_ROOT')


def extract(model, data, idx, batch=256):
    """Features (n, 512) and predictions (n,) of the clusters at positions idx, in float32 (no autocast)."""
    model.eval()
    feats, preds = [], []
    with torch.no_grad():
        for s in range(0, len(idx), batch):
            f = model.features(data.batch(idx[s:s + batch])).float()
            feats.append(f.cpu())
            preds.append(model.head(f).squeeze(1).cpu())
    return torch.cat(feats).numpy(), torch.cat(preds).numpy()


def write_csv(frame, path, **kw):
    """Write a CSV file under a temporary name, then rename it: nobody (another run) reads half a file."""
    tmp = f'{path}.{uuid.uuid4().hex[:8]}.tmp'
    frame.to_csv(tmp, index=False, **kw)
    os.replace(tmp, path)


# check: at the start, the model of a row with other bands is the pretrained ViT on the six SR bands
# (TRAIN_FROM_SCRATCH: the projection of the other bands is random too, not zero)
with torch.no_grad():
    _bands = ROWS['landsat_wavelet6'][1]
    _m = LandsatViT(_bands).eval()
    if VIT_STATE is not None:
        _ref = timm.create_model('vit_small_patch16_224', in_chans=len(SR_BANDS), num_classes=0).eval()
        _ref.load_state_dict(VIT_STATE, strict=False)
        _x = torch.rand(2, len(_bands), PATCH, PATCH)
        assert torch.allclose(_m.features(_x), _ref(_x[:, :len(SR_BANDS)]), atol=1e-4)
        del _ref, _x
    else:
        assert float(_m.backbone.patch_embed.proj.extra.weight.abs().sum()) > 0
print(f'LandsatViT: {sum(p.numel() for p in _m.parameters()) / 1e6:.1f} M parameters for {len(_bands)} bands; '
      + ('at the start it is the pretrained ViT on the six SR bands' if VIT_STATE is not None else
         'random start (TRAIN_FROM_SCRATCH)'))
del _m

In [ ]:
META = ['cluster_id', 'country', 'year', 'rural', 'lat', 'lon', 'nl_sensor', 'fold']    # columns of features.csv


def run_fold(key, data, y, folds, test_fold):
    """Train (or continue) the model of one row and test fold, then write the features and the prediction of every
    cluster (features.csv, all sets), the test predictions and result.json. y: IWI at the positions of data.rows
    (NaN: none); folds: their folds of 03 ('': none). Returns the test metrics (also in result.json)."""
    run_dir = os.path.join(RUN_ROOT, key, f'test_{test_fold}')
    result_file, features_file = os.path.join(run_dir, 'result.json'), os.path.join(run_dir, 'features.csv')
    if os.path.exists(result_file):
        with open(result_file) as f:
            done = json.load(f)
        check_start(done.get('from_scratch', False), run_dir)
        if os.path.exists(features_file):
            return done
    os.makedirs(run_dir, exist_ok=True)
    k = FOLDS.index(test_fold)
    val_fold = FOLDS[(k - 1) % len(FOLDS)]
    train_folds = [f for f in FOLDS if f not in (test_fold, val_fold)]
    has_y = np.isfinite(y)
    pos = lambda mask: torch.from_numpy(np.flatnonzero(mask)).to(DEVICE)
    tr = pos(has_y & np.isin(folds, train_folds))
    va, te = pos(has_y & (folds == val_fold)), pos(has_y & (folds == test_fold))
    y_dev = torch.tensor(np.nan_to_num(y), dtype=torch.float32, device=DEVICE)

    torch.manual_seed(SEED + k)
    model = LandsatViT(data.bands).to(DEVICE)
    with torch.no_grad():
        model.head.bias.fill_(float(y_dev[tr].mean()))
    opt = make_optimizer(model)
    gen = torch.Generator(device=DEVICE)
    gen.manual_seed(SEED * 100 + k)
    start, best, best_epoch = 0, float('inf'), -1
    last_file, best_file, hist_file = (os.path.join(run_dir, f) for f in ('last.pt', 'best.pt', 'history.csv'))
    if os.path.exists(result_file):                     # finished before: only features.csv is written again
        assert os.path.exists(best_file), f'{best_file} is missing: delete {result_file} to train this model again'
        start = EPOCHS
    elif os.path.exists(last_file):                     # continue a run that was stopped
        st = torch.load(last_file, map_location=DEVICE)
        check_start(st.get('from_scratch', False), run_dir)
        model.load_learned_state(st['model'])
        opt.load_state_dict(st['opt'])
        gen.set_state(st['gen'].cpu())
        start, best, best_epoch = st['epoch'] + 1, st['best'], st['best_epoch']
        if os.path.exists(hist_file):                   # epochs after the saved state are trained again
            hist = pd.read_csv(hist_file)
            hist[hist['epoch'] <= start].to_csv(hist_file, index=False)
        print(f'{key} test {test_fold}: continuing at epoch {start + 1}')
    elif os.path.exists(hist_file):
        os.remove(hist_file)                            # history of a run that stopped before saving its state

    for epoch in range(start, EPOCHS):
        t0 = time.time()
        model.train()
        perm = tr[torch.randperm(len(tr), generator=gen, device=DEVICE)]
        sq_sum = torch.zeros((), device=DEVICE)
        n_seen = 0
        n_steps = max(len(perm) // BATCH_SIZE, 1)
        for j, s in enumerate(range(0, len(perm) - BATCH_SIZE + 1, BATCH_SIZE)):    # last incomplete batch dropped
            for g in opt.param_groups:                          # warm-up and cosine decay, every step
                g['lr'] = g['base_lr'] * lr_factor(epoch + (j + 1) / n_steps)
            b = perm[s:s + BATCH_SIZE]
            with autocast():
                pred = model(data.batch(b))
            mse = F.mse_loss(pred.float(), y_dev[b])
            opt.zero_grad(set_to_none=True)
            mse.backward()
            opt.step()
            sq_sum += mse.detach() * len(b)
            n_seen += len(b)
        p_va = predict(model, data, va).cpu().numpy()
        y_va = y[va.cpu().numpy()]
        val_mse = float(((p_va - y_va) ** 2).mean())
        val_loss = val_mse
        if val_loss < best:
            best, best_epoch = val_loss, epoch
            save({'model': model.learned_state(), 'epoch': epoch, 'from_scratch': TRAIN_FROM_SCRATCH}, best_file)
        lrs = {f"lr_{g['name']}": g['lr'] for g in opt.param_groups}
        rec = {'epoch': epoch + 1, 'train_mse': float(sq_sum) / max(n_seen, 1), 'val_mse': val_mse,
               'val_loss': val_loss, 'val_r2': r2_score(y_va, p_va), **lrs, 'seconds': round(time.time() - t0, 1)}
        pd.DataFrame([rec]).to_csv(hist_file, mode='a', header=not os.path.exists(hist_file), index=False)
        save({'model': model.learned_state(), 'opt': opt.state_dict(), 'gen': gen.get_state(), 'epoch': epoch,
              'best': best, 'best_epoch': best_epoch, 'from_scratch': TRAIN_FROM_SCRATCH}, last_file)
        if epoch == start or (epoch + 1) % 10 == 0 or epoch + 1 == EPOCHS:
            print(f"{key} test {test_fold} epoch {epoch + 1}/{EPOCHS}: train mse {rec['train_mse']:.2f}, val mse "
                  f"{val_mse:.2f}, val R2 {rec['val_r2']:.3f}, best epoch {best_epoch + 1}, {rec['seconds']} s")

    # the best weights: features and prediction of every cluster (all sets), test metrics
    st = torch.load(best_file, map_location=DEVICE)
    check_start(st.get('from_scratch', False), run_dir)
    model.load_learned_state(st['model'])
    feats, pred = extract(model, data, torch.arange(len(data.rows), device=DEVICE))
    role = np.select([np.isin(folds, train_folds), folds == val_fold, folds == test_fold], ['train', 'val', 'test'], '')
    n_set = {r: int((role == r).sum()) for r in ('train', 'val', 'test', '')}       # rows of features.csv per set
    out = pd.concat([clusters.iloc[data.rows][[c for c in META if c in clusters]].reset_index(drop=True),
                     pd.DataFrame({'role': role, TARGET: y, f'{TARGET}_pred': pred}),
                     pd.DataFrame(feats, columns=[f'{FEATURE_PREFIX}{j}' for j in range(feats.shape[1])])], axis=1)
    write_csv(out.sort_values('cluster_id'), features_file, float_format='%.7g')

    te_np, va_np = te.cpu().numpy(), va.cpu().numpy()
    p_te, y_te = pred[te_np], y[te_np]
    test_rows = clusters.iloc[data.rows[te_np]]
    write_csv(test_rows[['cluster_id', 'country', 'year', 'nl_sensor']].assign(y_true=y_te, y_pred=p_te),
              os.path.join(run_dir, 'test_predictions.csv'))
    result = {'key': key, 'model': 'vits16_scratch' if TRAIN_FROM_SCRATCH else f'vits16_moco_{VIT_MODE}',
              'bands': data.n_bands, 'test_fold': test_fold,
              'val_fold': val_fold,
              'train_folds': ','.join(train_folds), 'keras_name': KERAS_NAME[test_fold],
              'sensors': ','.join(NL_SENSORS), 'use_temp1': USE_TEMP1,
              'n_train': len(tr), 'n_val': len(va), 'n_test': len(te), 'n_features': len(out),
              'best_epoch': st['epoch'] + 1, 'epochs': EPOCHS, 'val_r2': r2_score(y[va_np], pred[va_np]),
              'test_r2': r2_score(y_te, p_te), 'test_rmse': float(np.sqrt(((p_te - y_te) ** 2).mean())),
              'test_mae': float(np.abs(p_te - y_te).mean()), 'features_train': n_set['train'],
              'features_val': n_set['val'], 'features_test': n_set['test'], 'from_scratch': TRAIN_FROM_SCRATCH}
    with open(result_file + '.tmp', 'w') as f:
        json.dump(result, f, indent=1)
    os.replace(result_file + '.tmp', result_file)
    if os.path.exists(last_file):
        os.remove(last_file)                            # only needed to continue a stopped run
    print(f"{key} test {test_fold} ({KERAS_NAME[test_fold]}): test R2 {result['test_r2']:.3f}, RMSE "
          f"{result['test_rmse']:.2f} (best epoch {result['best_epoch']})\n"
          f"    features of all sets -> {features_file}: train {n_set['train']} ({', '.join(train_folds)}), "
          f"val {n_set['val']} ({val_fold}), test {n_set['test']} ({test_fold})"
          + (f", no fold {n_set['']}" if n_set[''] else ''))
    return result

# 5. Train

The models train on the clusters with complete data, an IWI value and a fold of 03, and compute the features of all clusters with complete data. The bands of a row are loaded once and used for its five test folds. Rows and test folds come from `RUN_ROWS` and `RUN_FOLDS`.

In [ ]:
rows = np.flatnonzero(complete)                                # rows of `clusters` that get features
y = clusters[TARGET].to_numpy(dtype=np.float64)[rows]          # NaN: no IWI, not trained on
folds = clusters['fold'].fillna('').astype(str).to_numpy()[rows]
trainable = np.isfinite(y) & np.isin(folds, FOLDS)
print(f'{len(rows)} clusters with complete data, {trainable.sum()} of them with {TARGET} and a fold:',
      pd.Series(folds[trainable]).value_counts().sort_index().to_dict())


def finished(key, f):
    run_dir = os.path.join(RUN_ROOT, key, f'test_{f}')
    return all(os.path.exists(os.path.join(run_dir, n)) for n in ('result.json', 'features.csv'))


if todo and DEVICE.type != 'cuda':
    print('WARNING: no GPU, training will be very slow')
for key in todo:
    desc, bands = ROWS[key]
    missing = [f for f in RUN_FOLDS if not finished(key, f)]
    if not missing:
        print(f'{key}: test folds {RUN_FOLDS} already done')
        continue
    print(f'\n=== {key}: {desc} ({len(bands)} bands), test folds {missing}')
    data = Bands(bands, rows)
    for f in missing:
        run_fold(key, data, y, folds, f)
    del data
    if DEVICE.type == 'cuda':
        torch.cuda.empty_cache()
if not todo:
    print('VITFEAT_ROWS=none: no training, only the results')

# 6. Results and the features of every cluster

For every row: the mean and standard deviation of the test R² over the models that are done, and the R², RMSE and MAE (in IWI points) of all test predictions pooled, also for each period. Next to them are the mean test R² of ResNet-18 on the same row (09, `RESNET_RUNS`) and of the NL-only models (08, `NL_RUNS`), when they are done. For every model: the test metrics and the number of clusters of each set in its `features.csv` (`features_train`, `features_val`, `features_test`; they include the clusters without IWI). The tables go to `landsat_vits16_results.csv` and `landsat_vits16_per_fold.csv` in `RUN_ROOT`. When the five models of a row are done, the test rows of their `features.csv` files are put together in `<key>/features_oof.csv`.

Reading the features, for example of row `landsat` and model `test_E` (Keras Fold12345), and merging them with those of the NL model of 08:

```python
feat = pd.read_csv(os.path.join(RUN_ROOT, 'landsat', 'test_E', 'features.csv'))
X = feat.filter(like=FEATURE_PREFIX).to_numpy()          # (clusters, 384)
train, test = feat['role'].isin(['train', 'val']), feat['role'] == 'test'
nl = pd.read_csv(os.path.join(NL_RUNS, 'test_E', 'features.csv'))
both = feat.merge(nl[['cluster_id'] + [c for c in nl if c.startswith('nl_feat_')]], on='cluster_id')
```

In [ ]:
def scores(p):
    err = p['y_pred'] - p['y_true']
    return {'clusters': len(p), 'R2': r2_score(p['y_true'], p['y_pred']) if len(p) > 1 else np.nan,
            'RMSE': float(np.sqrt((err ** 2).mean())), 'MAE': float(err.abs().mean())}


def fold_results(key):
    """result.json of every finished test fold of a row, and its pooled test predictions (or None)."""
    found, preds = [], []
    for f in FOLDS:
        run_dir = os.path.join(RUN_ROOT, key, f'test_{f}')
        if os.path.exists(os.path.join(run_dir, 'result.json')):
            with open(os.path.join(run_dir, 'result.json')) as fh:
                found.append(json.load(fh))
            check_start(found[-1].get('from_scratch', False), run_dir)   # a run folder holds one kind of run
            preds.append(pd.read_csv(os.path.join(run_dir, 'test_predictions.csv')))
    return found, (pd.concat(preds, ignore_index=True) if preds else None)


per_fold, rows_out = [], []
for key, (desc, bands) in ROWS.items():
    found, p = fold_results(key)
    per_fold += found
    r = pd.Series([x['test_r2'] for x in found], dtype=float)
    out = {'key': key, 'input': desc, 'bands': len(bands), 'folds done': len(r),
           'test R2 mean': r.mean() if len(r) else np.nan, 'test R2 sd': r.std() if len(r) > 1 else np.nan,
           'test R2 min': r.min() if len(r) else np.nan, 'test R2 max': r.max() if len(r) else np.nan}
    if p is not None:
        sc = scores(p)
        out.update({'pooled R2': sc['R2'], 'pooled RMSE': sc['RMSE'], 'pooled MAE': sc['MAE']})
        for s, ps in p.groupby('nl_sensor'):
            out[f'pooled R2 {s}'] = scores(ps)['R2']
    rows_out.append(out)
# ResNet-18 on the same rows (09) for comparison: the mean test R2 of its models
resnet_file = os.path.join(RESNET_RUNS, 'landsat_resnet18_results.csv')
RESNET_R2 = (pd.read_csv(resnet_file).set_index('key')['test R2 mean'].to_dict()
             if os.path.exists(resnet_file) else {})
for out in rows_out:
    out['ResNet-18 R2 (09)'] = RESNET_R2.get(out['key'], np.nan)
RES_COLS = ['key', 'input', 'bands', 'folds done', 'test R2 mean', 'test R2 sd', 'test R2 min', 'test R2 max',
            'pooled R2', 'pooled RMSE', 'pooled MAE'] + [f'pooled R2 {s}' for s in NL_SENSORS] + \
           ['ResNet-18 R2 (09)']
results = pd.DataFrame(rows_out, columns=RES_COLS)
FOLD_COLS = ['key', 'test_fold', 'val_fold', 'train_folds', 'keras_name', 'n_train', 'n_val', 'n_test', 'best_epoch',
             'val_r2', 'test_r2', 'test_rmse', 'test_mae', 'features_train', 'features_val', 'features_test']
per_fold = pd.DataFrame(per_fold, columns=FOLD_COLS + ['model', 'bands', 'n_features', 'sensors', 'use_temp1',
                                                    'epochs'])
os.makedirs(RUN_ROOT, exist_ok=True)
write_csv(results, os.path.join(RUN_ROOT, 'landsat_vits16_results.csv'))
write_csv(per_fold, os.path.join(RUN_ROOT, 'landsat_vits16_per_fold.csv'))

# NL only (08) for comparison: the mean test R2 of its models
NL_REF = None
nl_file = os.path.join(NL_RUNS, 'nl_resnet18_results.csv')
if os.path.exists(nl_file):
    nl_res = pd.read_csv(nl_file)
    nl_res = nl_res[nl_res['what'].str.startswith('mean of')]
    if len(nl_res):
        NL_REF = (float(nl_res['R2'].iloc[0]), nl_res['what'].iloc[0])

# features_oof.csv of every row whose five models are done
for key in ROWS:
    files = [os.path.join(RUN_ROOT, key, f'test_{f}', 'features.csv') for f in FOLDS]
    if not all(os.path.exists(p) for p in files):
        continue
    oof_file = os.path.join(RUN_ROOT, key, 'features_oof.csv')
    if os.path.exists(oof_file) and os.path.getmtime(oof_file) >= max(map(os.path.getmtime, files)):
        continue
    oof = pd.concat([pd.read_csv(p).query("role == 'test'") for p in files], ignore_index=True)
    assert oof['cluster_id'].is_unique
    write_csv(oof.sort_values('cluster_id'), oof_file, float_format='%.7g')
    print(f'{key}: features of {len(oof)} clusters, each from the model that did not see its fold -> {oof_file}')
print('rows with features_oof.csv:',
      [k for k in ROWS if os.path.exists(os.path.join(RUN_ROOT, k, 'features_oof.csv'))] or
      'none yet (written when the five models of a row are done)')

print(f'\n{VIT_NAME} on Landsat-based inputs ({" + ".join(NL_SENSORS)} periods), '
      f'{TARGET} target, spatial folds of 03; ResNet-18 (09): '
      + (f'{len(RESNET_R2)} rows' if RESNET_R2 else f'no results in {RESNET_RUNS}') + '; NL only (08): '
      + (f'test R2 {NL_REF[0]:.3f} ({NL_REF[1]})' if NL_REF else f'no results in {NL_RUNS}'))
display(per_fold[FOLD_COLS].round(3))
results.round(3)

In [ ]:
done = results[results['folds done'] > 0].iloc[::-1]
if len(done):
    fig, ax = plt.subplots(figsize=(9.5, 0.5 * len(done) + 1.6), constrained_layout=True)
    yy = np.arange(len(done))
    handles = [ax.barh(yy, done['test R2 mean'], xerr=done['test R2 sd'].fillna(0), height=0.6, color='#2a78d6',
                       error_kw={'ecolor': '#52514e', 'elinewidth': 1, 'capsize': 2},
                       label=f'ViT-S/16 {"random start" if TRAIN_FROM_SCRATCH else VIT_MODE}: mean ± sd of the '
                             'test folds')]
    if done['ResNet-18 R2 (09)'].notna().any():
        handles.append(ax.scatter(done['ResNet-18 R2 (09)'], yy, marker='o', s=40, color='#1baf7a',
                                  edgecolors='white', linewidths=1.2, zorder=3,
                                  label='ResNet-18, 09 (mean of the test folds)'))
    if NL_REF:
        handles.append(ax.axvline(NL_REF[0], color='#52514e', linewidth=1, linestyle='--',
                                  label=f'NL only, 08 ({NL_REF[0]:.2f})'))
    for k, v, e in zip(yy, done['test R2 mean'], done['test R2 sd'].fillna(0)):
        if v > 0.15:                                       # value inside the bar, clear of the markers
            ax.text(0.01, k, f'{v:.2f}', va='center', ha='left', color='white', fontsize=8)
        else:                                              # small or negative: right of the error bar
            ax.text(min(max(v + e, 0), 0.85) + 0.015, k, f'{v:.2f}', va='center', ha='left',
                    color='#52514e', fontsize=8)
    ax.set_yticks(yy)
    ax.set_yticklabels(done['key'])
    lo = np.nanmin(np.r_[done['test R2 mean'].to_numpy(float), done['ResNet-18 R2 (09)'].to_numpy(float)])
    ax.set_xlim(max(min(0, lo - 0.05), -1), 1)                            # a row below -1 is cut at -1
    ax.set_xlabel('test R²')
    ax.grid(axis='x', color='#e1e0d9', linewidth=0.8)
    ax.set_axisbelow(True)
    for side in ('top', 'right'):
        ax.spines[side].set_visible(False)
    ax.legend(handles=handles, loc='upper center', bbox_to_anchor=(0.5, -0.1 - 0.6 / len(done)), ncol=2,
              frameon=False, fontsize=8)
    ax.set_title(f'{VIT_NAME} on Landsat-based inputs, {TARGET.upper()} of the DHS '
                 'clusters, spatial 5-fold CV', loc='left', fontsize=11)
    png = os.path.join(RUN_ROOT, 'landsat_vits16_r2.png')
    fig.savefig(png + '.tmp.png', dpi=150)
    os.replace(png + '.tmp.png', png)
    plt.show()
else:
    print('no finished models yet')